# 自定义中间件-Node-style hooks

## 1.基本用法

### 1.1基于装饰器的实现

In [ ]:
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model, after_model, before_agent, after_agent
from typing import Any

@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_model <----"
    return None   # 不干扰主进程

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_model <----"
    return None


@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> before_agent <----"
    return None


@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----> after_agent <----"
    return None

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

In [ ]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model = model,
    middleware = [
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware
    ]
)

res = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in res["messages"]:
    msg.pretty_print()

### 1.2基于类的实现

In [12]:
from langchain.agents.middleware import AgentMiddleware

class MyMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_model <----"
        return None   # 不干扰主进程


    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_model <----"
        return None



    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_agent <----"
        return None


    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> after_agent <----"
        return None

In [13]:
my_middleware = MyMiddleware()

agent = create_agent(
    model = model,
    middleware = [my_middleware]
)

res = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in res["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好----> before_agent <--------> before_model <----
================================== Ai Message ==================================

你好！如果你说的是 **LangChain / LangGraph Agent 的 middleware hook**，那大致流程是：

```text
用户输入 “你好”
   ↓
before_agent   // 每次 Agent 运行只执行一次
   ↓
before_model   // 每次调用 LLM 前都执行
   ↓
model
   ↓
after_model
   ↓
如果有 tool calls → tools → before_model → model → ...
   ↓
after_agent    // Agent 结束时执行
   ↓
最终输出
```

关键区别：

- **before_agent**
  - 时机：整个 Agent 开始前，通常只跑一次。
  - 作用域：整个 run。
  - 适合：初始化 state、注入全局 system prompt、鉴权、埋点、准备上下文。

- **before_model**
  - 时机：每一次调用 LLM 之前。
  - 作用域：单次 model call。
  - 适合：裁剪/改写 messages、动态加 RAG 上下文、临时 system prompt、限流、按轮次注入信息。

所以它们不是并列关系：

```text
before_agent → before_model → model → after_model → ... → after_agent
```

`before_agent` 先于第一次 `before_model`，但 `before_model` 可能在一个 Agent run 里执行很多次。

你的“你好”这条消息通常会先进入 `state.messages`，然后 `before_agent` 可以看到并修改它，之后 `